# FSPD Exploratory Data Analysis
## Analyzing the Food System Policy Database for ML Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
df = pd.read_csv('../data/FSPD.csv', low_memory=False)
print(f'Loaded {len(df):,} policies')
df.head()

## 1. Dataset Overview

In [ ]:
print('Dataset Shape:', df.shape)
print('\nColumn Names:')
for i, col in enumerate(df.columns, 1):
    print(f'{i:2d}. {col}')

## 2. Classification Targets Analysis

In [ ]:
# Segment distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Segment
segment_counts = df['segment'].value_counts()
axes[0, 0].barh(range(len(segment_counts)), segment_counts.values)
axes[0, 0].set_yticks(range(len(segment_counts)))
axes[0, 0].set_yticklabels(segment_counts.index)
axes[0, 0].set_xlabel('Count')
axes[0, 0].set_title(f'Segment Distribution ({len(segment_counts)} categories)')
axes[0, 0].invert_yaxis()

# Type (itype)
itype_counts = df['itype'].value_counts()
axes[0, 1].barh(range(len(itype_counts)), itype_counts.values)
axes[0, 1].set_yticks(range(len(itype_counts)))
axes[0, 1].set_yticklabels(itype_counts.index)
axes[0, 1].set_xlabel('Count')
axes[0, 1].set_title(f'Type Distribution ({len(itype_counts)} categories)')
axes[0, 1].invert_yaxis()

# Objectives
objectives = ['inclusion', 'envitarget', 'diethealth']
obj_counts = [df[obj].sum() for obj in objectives]
axes[1, 0].bar(objectives, obj_counts)
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Objective Flags (Binary Targets)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Source database
db_counts = df['db'].value_counts()
axes[1, 1].bar(range(len(db_counts)), db_counts.values)
axes[1, 1].set_xticks(range(len(db_counts)))
axes[1, 1].set_xticklabels(db_counts.index, rotation=45)
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Source Database Distribution')

plt.tight_layout()
plt.savefig('../outputs/plots/target_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Lever Analysis (Detailed)

In [ ]:
# Top 30 levers
lever_counts = df['lever'].value_counts().head(30)

plt.figure(figsize=(12, 10))
plt.barh(range(len(lever_counts)), lever_counts.values)
plt.yticks(range(len(lever_counts)), lever_counts.index, fontsize=9)
plt.xlabel('Count')
plt.title(f'Top 30 Policy Levers (out of {df["lever"].nunique()} total)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/plots/lever_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Total unique levers: {df["lever"].nunique()}')
print(f'\nLevers with <10 examples: {(df["lever"].value_counts() < 10).sum()}')
print(f'Levers with <5 examples: {(df["lever"].value_counts() < 5).sum()}')

## 4. Text Analysis

In [ ]:
# Text length distribution
df['text_length'] = df['policydecision_details'].fillna('').str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['text_length'], bins=50, edgecolor='black')
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Policy Text Lengths')
axes[0].axvline(df['text_length'].median(), color='red', linestyle='--', label=f'Median: {df["text_length"].median():.0f}')
axes[0].legend()

# Box plot
axes[1].boxplot(df['text_length'])
axes[1].set_ylabel('Text Length (characters)')
axes[1].set_title('Text Length Distribution (Box Plot)')

plt.tight_layout()
plt.savefig('../outputs/plots/text_length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('Text Length Statistics:')
print(df['text_length'].describe())

## 5. Geographic Coverage

In [ ]:
# Top countries
country_counts = df['country'].value_counts().head(20)

plt.figure(figsize=(12, 8))
plt.barh(range(len(country_counts)), country_counts.values)
plt.yticks(range(len(country_counts)), country_counts.index)
plt.xlabel('Number of Policies')
plt.title(f'Top 20 Countries by Policy Count (out of {df["country"].nunique()} total)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/plots/country_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Class Imbalance Analysis

In [ ]:
def calculate_imbalance_ratio(series):
    """Calculate imbalance ratio: max_class / min_class"""
    counts = series.value_counts()
    return counts.max() / counts.min()

print('Class Imbalance Ratios:')
print(f'  Segment: {calculate_imbalance_ratio(df["segment"]):.1f}:1')
print(f'  Lever: {calculate_imbalance_ratio(df["lever"]):.1f}:1')
print(f'  Type: {calculate_imbalance_ratio(df["itype"]):.1f}:1')

print('\nObjective Distributions:')
for obj in ['inclusion', 'envitarget', 'diethealth']:
    positive_pct = df[obj].mean() * 100
    print(f'  {obj}: {positive_pct:.1f}% positive')

## 7. Prepare Data for ML

In [ ]:
# Filter out rows with missing targets or text
ml_df = df[df['policydecision_details'].notna()].copy()
ml_df = ml_df[ml_df['segment'].notna()]
ml_df = ml_df[ml_df['lever'].notna()]
ml_df = ml_df[ml_df['itype'].notna()]

print(f'Policies suitable for ML: {len(ml_df):,} / {len(df):,} ({len(ml_df)/len(df)*100:.1f}%)')

# Save cleaned dataset
ml_df.to_csv('../data/FSPD_ml_ready.csv', index=False)
print('\nSaved ML-ready dataset to data/FSPD_ml_ready.csv')

## 8. Sample Policies by Category

In [ ]:
# Show examples from each segment
print('Sample policies from each segment:\n')
for segment in df['segment'].unique()[:5]:
    sample = df[df['segment'] == segment].iloc[0]
    print(f'SEGMENT: {segment}')
    print(f'Lever: {sample["lever"]}')
    print(f'Type: {sample["itype"]}')
    print(f'Text: {sample["policydecision_details"][:200]}...')
    print('-' * 80)
    print()